# Exercise Week 4: A* Search Algorithm

#### MEC-E5012 - Vehicle Mechatronics: Control D - Autumn 2025

## Introduction:

In this exercise, you will **implement the A\* (A-star) pathfinding algorithm** and visualize its progress in a maze using Python.

`A* search` is  one of the most popular techniques used in path finding and graph traversals. What makes A* search algorithm powerful is that at each step it picks the node to explore in the next step based on `f` value, which is the sum of two other parameters - `g` and `h`. At each step, it picks the node with lowest `f` value, and process that node/cell.  

The core idea behind A\* is to always expand the node with the lowest total estimated cost, defined as:
`f(n) = g(n) + h(n)`

where:

- `g(n)`: The cumulative cost of the shortest known path from the start node to the current node. This cost is typically computed as the sum of the edge weights along the path that has been found so far.
- `h(n)`: the estimated movement cost to move from that given square on the grid to the final destination. This is often referred to as the heuristic, which is nothing but a kind of smart guess. We really don’t know the actual distance until we find the path, because all sorts of things can be in the way (walls, water, etc.).  If the heuristic is admissible, then A* search is optimal.
- `f(n)`: Total estimated cost through the current node 

You can use different heuristic functions in A*, such as Manhattan, Euclidean,  Chebyshev distance and more. In this exercise, we’ll use the Manhattan distance, but feel free to try other heuristics and see how the search behaves in the maze.

You can use the Manhattan Distance from the agent's current position to the singular waypoint as the A* heuristic function. For two grid coordinates a and b, the manhattan distance is given by: `abs(a[0] - b[0]) + abs(a[1] - b[1])`

*(Animation from the course material, not included in this repository.)*


The automation was created at this <a href="https://qiao.github.io/PathFinding.js/visual/">website</a>. Feel free to play around with it, and also try some different Heuristic Function.

In the animation above, the A* algorithm is used to search for a path from the green starting point to the red waypoint, using the Manhattan Distance as the heuristic. The grey points represent walls or obstacles. The blue points indicate the nodes that have already been explored, while the light green points show the frontier nodes that are currently under consideration.

### In this exercise, you will:

- **Implement** the core logic of the A\* algorithm, one small step at a time
- **Visualize** the search process and the discovered path on a 2D grid maze
- **Experiment** with different maze layouts, start/goal positions, and estimates (heuristics)
- **Observe** how the choice of estimate affects how much of the maze is searched and the path found

---

Once you finish the implementation, try modifying the maze and estimate, and see how A\* adapts!

Don't forget to press **SPACE** to start the simulation.

In [1]:
# Run this once to install the visualization library (safe to re-run).
!pip install pygame


## The idea of A\* in plain words (read this first)

Imagine standing in the maze. A\* keeps a list of cells it has **found but not yet
visited** — we call this the **frontier**. Over and over, it asks one simple question:

> **Which frontier cell looks most promising?  Visit that one next.**

"Most promising" means the smallest value of

**priority  =  cost so far  +  estimate to the goal**

- **cost so far** — how many steps it took to get to that cell (we call it `cost_so_far`).
- **estimate to the goal** — a *guess* of how far the goal still is (your `heuristic`, Task 1).

That is the whole algorithm. In everyday language:

```
put the start cell in the frontier
repeat:
    current = the frontier cell with the smallest priority
    if current is the goal  ->  we're done!
    otherwise, look at the cells next to current, and for each one:
        if we've never seen it, or we just found a cheaper way to reach it:
            remember its cost, remember which cell we came from,
            and add it to the frontier
```

You only fill in **three small pieces**. Everything else is written for you:

| Task | Points | What you write |
|------|:------:|----------------|
| **Task 1** | 1 p | `heuristic(a, b)` — the *estimate to the goal* |
| **Task 2** | 5 p | inside the neighbor loop: decide when to keep a neighbor, and remember it |
| **Task 3** | 4 p | at the end of `astar`: follow `came_from` backwards to build the route |

Before implementing the A* algorithm, make sure you understand how the variables used in the code correspond to the concepts introduced in the Exercise 4 Guide.

| Variable in the code | A* concept | Meaning |
|---|---|---|
| `open_set` | Priority queue | Stores candidate nodes and determines which node should be expanded next. |
| `frontier` | Frontier | Nodes that have been discovered but not yet expanded. |
| `visited` | Expanded/visited set | Nodes that have already been expanded. |
| `g_score[node]` | $g(n)$ | The best known cost from the start node to `node`. |
| `heuristic(node, goal)` | $h(n)$ | The estimated cost from `node` to the goal. |
| `g + h` | $f(n)$ | The estimated total cost of a path through the node: $f(n)=g(n)+h(n)$. |
| `came_from[node]` | Parent link | Stores the node from which `node` was reached. This is later used to reconstruct the final path. |

Run the cells one by one, from top to bottom. When you reach a maze cell, a window
opens — **press SPACE** to watch A\* search.

In [2]:
# ---- Given: imports, colors and window setup (you do NOT need to edit this) ----
import pygame
import time

CELL_PIXELS = 40      # preferred size of one grid cell, in pixels
MAX_WINDOW  = 640     # target max window size; cells shrink to fit large mazes

# Colors (R, G, B)
WHITE  = (255, 255, 255)
BLACK  = (0, 0, 0)
GREY   = (200, 200, 200)
GREEN  = (0, 255, 0)
PURPLE = (128, 0, 128)
BLUE   = (0, 200, 255)
YELLOW = (255, 255, 0)

pygame.init()

def make_screen(maze):
    """Create a pygame window sized to fit `maze`; return (screen, cell_pixels)."""
    rows, cols = len(maze), len(maze[0])
    cell = max(6, min(CELL_PIXELS, MAX_WINDOW // max(rows, cols)))
    screen = pygame.display.set_mode((cols * cell, rows * cell))
    pygame.display.set_caption("A* Pathfinding: Practical Exercise")
    return screen, cell


pygame 2.6.1 (SDL 2.28.4, Python 3.10.10)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [3]:
# ---- Given helpers: you do NOT need to edit or fully understand this cell ----
# (But do read pick_lowest_priority - it is the heart of A*.)

def neighbors(maze, cell):
    """Return the free cells directly above, below, left and right of `cell`."""
    rows, cols = len(maze), len(maze[0])
    row, col = cell
    result = []
    for drow, dcol in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        r, c = row + drow, col + dcol
        if 0 <= r < rows and 0 <= c < cols and maze[r][c] == 0:   # inside the grid and not a wall
            result.append((r, c))
    return result


def pick_lowest_priority(frontier, goal, cost_so_far):
    """Return the frontier cell with the smallest priority.

        priority = cost_so_far[cell] + heuristic(cell, goal)

    This is THE core rule of A*: always visit the most promising cell next.

    Tie-break: if two cells share the same priority (same f cost), we prefer the
    one with the smaller heuristic - i.e. the cell that looks closer to the goal.
    (This matches the pen-and-paper example in the Exercise 4 Guide.)
    """
    best_cell = None
    best_key = None
    for cell in frontier:
        h = heuristic(cell, goal)
        priority = cost_so_far[cell] + h
        key = (priority, h)                 # compare f first, then use h as tie-breaker
        if best_key is None or key < best_key:
            best_cell = cell
            best_key = key
    return best_cell


def remember_step(steps, explored, frontier, current, cost, estimate, path=None):
    """Save a snapshot of the search so the animation can replay it later."""
    steps.append((list(explored), list(frontier), current, cost, estimate, path))


def draw_maze(screen, maze, cell_px, start, goal,
              path=(), explored=(), frontier=(), priorities=None, font=None):
    """Draw one frame: purple = start, green = goal/path, yellow = frontier,
    blue = explored, black = wall, white = free cell."""
    rows, cols = len(maze), len(maze[0])
    path, explored, frontier = set(path), set(explored), set(frontier)
    for r in range(rows):
        for c in range(cols):
            rect = pygame.Rect(c * cell_px, r * cell_px, cell_px, cell_px)
            if (r, c) == start:
                color = PURPLE
            elif (r, c) == goal:
                color = (93, 204, 49)
            elif (r, c) in path:
                color = GREEN
            elif (r, c) in frontier:
                color = YELLOW
            elif (r, c) in explored:
                color = BLUE
            elif maze[r][c] == 1:
                color = BLACK
            else:
                color = WHITE
            pygame.draw.rect(screen, color, rect)
            pygame.draw.rect(screen, GREY, rect, 1)
            if font and priorities and (r, c) in priorities and (r, c) not in (start, goal):
                label = font.render(str(priorities[(r, c)]), True, BLACK)
                screen.blit(label, (c * cell_px + 3, r * cell_px + 2))


In [4]:
# ===========================  TASK 1  (1 point)  ============================
# Write the "estimate to the goal" - the heuristic h.

def heuristic(a, b):
    """Guess how far apart two cells are, ignoring the walls.

    We use the *Manhattan distance*: how many steps you would take if you could
    only move up/down/left/right and there were no walls. For two cells
    a = (row_a, col_a) and b = (row_b, col_b) that is:

        abs(row_a - row_b) + abs(col_a - col_b)
    """

    # ---- YOUR CODE HERE (1 p): return the Manhattan distance ----
    # Calculate absolute differences for rows and columns
    return abs(a[0] - b[0]) + abs(a[1] - b[1])
    # -------------------------------------------------------------


In [5]:
# =====================  TASKS 2 & 3  -  the A* search  ======================
# Read the comments carefully - you only fill in two short spots (marked below).

def astar(maze, start, goal):
    frontier    = [start]        # cells we have found but not visited yet
    explored    = []             # cells we have already visited
    came_from   = {}             # came_from[cell] = the cell we arrived from
    cost_so_far = {start: 0}     # cost_so_far[cell] = number of steps from start
    steps       = []             # snapshots for the animation (given - just ignore it)

    while frontier:
        # THE CORE RULE: visit the most promising frontier cell next
        # (smallest  cost_so_far + heuristic).
        current = pick_lowest_priority(frontier, goal, cost_so_far)
        frontier.remove(current)
        explored.append(current)
        remember_step(steps, explored, frontier, current,
                      cost_so_far[current], heuristic(current, goal))   # given

        if current == goal:
            break                # reached the goal - stop searching

        # Look at each cell next to `current`
        for neighbor in neighbors(maze, current):
            new_cost = cost_so_far[current] + 1      # one more step than current

            # -------------------- TASK 2  (5 points) --------------------
            # Keep `neighbor` if EITHER of these is true:
            #    * we have never seen it   ->  neighbor not in cost_so_far
            #    * OR this new route is cheaper  ->  new_cost < cost_so_far[neighbor]
            #
            # When you keep it, do THREE things:
            #    (a) store the cost:            cost_so_far[neighbor] = new_cost
            #    (b) remember where it came from: came_from[neighbor] = current
            #    (c) add it to the frontier, but only if it is not already there:
            #            if neighbor not in frontier:
            #                frontier.append(neighbor)
            #
            # Replace the line `pass` below with your code.

            # Check if neighbor is unvisited or we found a cheaper path
            if neighbor not in cost_so_far or new_cost < cost_so_far[neighbor]:
                # (a) Update the best known cost to reach this neighbor
                cost_so_far[neighbor] = new_cost
                
                # (b) Record the path for backtracking
                came_from[neighbor] = current
                
                # (c) Add to frontier to explore it later, if not already present
                if neighbor not in frontier:
                    frontier.append(neighbor)

            # ------------------------------------------------------------

    # If the goal was never reached, there is no path.
    if goal not in came_from and goal != start:
        remember_step(steps, explored, frontier, start, 0, 0, [])   # given
        return steps

    # -------------------- TASK 3  (4 points) --------------------
    # Build the path from start to goal using `came_from`.
    # `came_from[cell]` tells you which cell you arrived from, so you can walk
    # backwards from the goal to the start.
    #
    # Steps to follow:
    #    1. make an empty list called `path`
    #    2. start at `cell = goal`
    #    3. while cell is not the start:  add cell to path, then go to came_from[cell]
    #    4. add the start cell too
    #    5. reverse the list so it goes start -> goal   (use  path.reverse())

    path = []
    cell = goal
    
    # Backtrack from goal to start using came_from dictionary
    while cell != start:
        path.append(cell)
        cell = came_from[cell]
        
    # Append the starting node and reverse the list for correct order
    path.append(start)
    path.reverse()

    # ------------------------------------------------------------

    remember_step(steps, explored, frontier, goal, cost_so_far.get(goal, 0), 0, path)  # given
    return steps


In [6]:
# ---- Given: the animation window (you do NOT need to edit this) ----
# It calls YOUR astar() and replays the snapshots. Press SPACE to start.

def run_visualization(maze, start, goal, anim_speed=0.1, show_numbers=True):
    """Open a window and animate A* on `maze`. Press SPACE to run the search.

    While searching, it prints each visited cell together with its cost, estimate
    and priority; on small mazes the priority is also drawn inside the cell. When
    the goal is reached the green path is shown and a short summary is printed.
    """
    pygame.init()          # (re)initialize - a previous run may have called pygame.quit()
    screen, cell_px = make_screen(maze)
    font = pygame.font.SysFont(None, max(12, cell_px // 2)) if show_numbers else None
    show_overlay = font is not None and len(maze[0]) <= 20     # only readable on small mazes

    steps = []
    i = 0
    started = False
    summary_printed = False
    path = []
    explored, frontier, priorities = [], [], {}

    running = True
    while running:
        screen.fill(WHITE)
        draw_maze(screen, maze, cell_px, start, goal, path, explored, frontier,
                  priorities if show_overlay else None, font)
        pygame.display.flip()

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            elif event.type == pygame.KEYDOWN and event.key == pygame.K_SPACE and not started:
                steps = astar(maze, start, goal)      # run the whole search now
                started, i = True, 0
                explored, frontier, priorities, path = [], [], {}, []

        if started and i < len(steps):
            explored, frontier, current, cost, estimate, maybe_path = steps[i]
            priorities[current] = cost + estimate
            print(f"step {i + 1:3}: visit {current}   "
                  f"cost={cost}  estimate={estimate}  priority={cost + estimate}")
            if maybe_path is not None:
                path = maybe_path
            i += 1
            time.sleep(anim_speed)
        elif started and not summary_printed:
            summary_printed = True
            if path:
                print("\n--- Search finished ---")
                print(f"Path length (steps) : {len(path) - 1}")
                print(f"Cells visited       : {len(explored)}")
            else:
                print("\nNo path found.")

    pygame.quit()


In [7]:
# ---- Self-check: run this AFTER you finish Tasks 1, 2 and 3 ----
# It quietly runs your code on a tiny maze (no window) and tells you if it works.

def find_path(maze, start, goal):
    """Run the search and return just the final path (the last saved snapshot)."""
    steps = astar(maze, start, goal)
    return steps[-1][-1]        # the path is the last item of the last snapshot

# Task 1: the heuristic
assert heuristic((0, 0), (3, 4)) == 7, "Task 1: heuristic should be the Manhattan distance"
assert heuristic((2, 2), (2, 2)) == 0, "Task 1: distance from a cell to itself must be 0"

# Tasks 2 & 3: a small maze you can check by hand
small_maze = [
    [0, 0, 0, 0],
    [1, 1, 0, 1],
    [0, 0, 0, 0],
    [0, 1, 1, 0],
]
path = find_path(small_maze, (0, 0), (3, 3))
assert path,                "Tasks 2/3: no path found - is the frontier being filled?"
assert path[0]  == (0, 0),  "Task 3: the path should start at the start cell"
assert path[-1] == (3, 3),  "Task 3: the path should end at the goal cell"
assert len(path) - 1 == 6,  f"Tasks 2/3: shortest path is 6 steps, but you got {len(path) - 1}"

print("All checks passed - nice work! You can now run the mazes below.")


All checks passed - nice work! You can now run the mazes below.


In [8]:
# Maze 1  (15 x 15).  Press SPACE inside the window to start the search.

maze = [
    [0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0],
    [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0],
    [0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0],
    [0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0],
    [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0],
    [0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0],
    [1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0],
    [0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0],
    [1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1],
    [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0],
    [0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0],
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0],
]

start = (2, 4)
goal  = (6, 10)
# NOTE: change start/goal to any free (0) cells to see different searches.

run_visualization(maze, start, goal, anim_speed=0.1)   # smaller anim_speed = faster


step   1: visit (2, 4)   cost=0  estimate=10  priority=10
step   2: visit (2, 5)   cost=1  estimate=9  priority=10
step   3: visit (2, 6)   cost=2  estimate=8  priority=10
step   4: visit (1, 6)   cost=3  estimate=9  priority=12
step   5: visit (1, 4)   cost=1  estimate=11  priority=12
step   6: visit (0, 6)   cost=4  estimate=10  priority=14
step   7: visit (0, 7)   cost=5  estimate=9  priority=14
step   8: visit (0, 8)   cost=6  estimate=8  priority=14
step   9: visit (1, 8)   cost=7  estimate=7  priority=14
step  10: visit (2, 8)   cost=8  estimate=6  priority=14
step  11: visit (2, 9)   cost=9  estimate=5  priority=14
step  12: visit (2, 10)   cost=10  estimate=4  priority=14
step  13: visit (0, 4)   cost=2  estimate=12  priority=14
step  14: visit (1, 10)   cost=11  estimate=5  priority=16
step  15: visit (2, 11)   cost=11  estimate=5  priority=16
step  16: visit (0, 3)   cost=3  estimate=13  priority=16
step  17: visit (0, 10)   cost=12  estimate=6  priority=18
step  18: visit (2

In [9]:
# Maze 2  (41 x 41).  Same one-line call - the window is sized automatically.

maze = [
    [0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0],
    [0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0],
    [0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0],
    [0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0],
    [1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0],
    [0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0],
    [0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0],
    [0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1],
    [0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0],
    [0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0],
    [0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0],
    [0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0],
    [0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0],
    [0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1],
    [0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0],
    [0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0],
    [0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0],
    [1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1],
    [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0],
    [0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0],
    [0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0],
    [0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0],
    [0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0],
    [0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0],
    [0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0],
    [0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1],
    [0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0],
    [0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0],
    [0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0],
    [0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0],
    [0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0],
    [0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0],
    [0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0],
    [0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0],
    [0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    [1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
]

start = (0, 0)
goal  = (40, 40)

run_visualization(maze, start, goal, anim_speed=0.05)


step   1: visit (0, 0)   cost=0  estimate=80  priority=80
step   2: visit (1, 0)   cost=1  estimate=79  priority=80
step   3: visit (2, 0)   cost=2  estimate=78  priority=80
step   4: visit (3, 0)   cost=3  estimate=77  priority=80
step   5: visit (4, 0)   cost=4  estimate=76  priority=80
step   6: visit (4, 1)   cost=5  estimate=75  priority=80
step   7: visit (4, 2)   cost=6  estimate=74  priority=80
step   8: visit (3, 2)   cost=7  estimate=75  priority=82
step   9: visit (2, 2)   cost=8  estimate=76  priority=84
step  10: visit (1, 2)   cost=9  estimate=77  priority=86
step  11: visit (0, 2)   cost=10  estimate=78  priority=88
step  12: visit (0, 3)   cost=11  estimate=77  priority=88
step  13: visit (0, 4)   cost=12  estimate=76  priority=88
step  14: visit (1, 4)   cost=13  estimate=75  priority=88
step  15: visit (2, 4)   cost=14  estimate=74  priority=88
step  16: visit (2, 5)   cost=15  estimate=73  priority=88
step  17: visit (2, 6)   cost=16  estimate=72  priority=88
step  1

step 137: visit (24, 14)   cost=130  estimate=42  priority=172
step 138: visit (24, 13)   cost=131  estimate=43  priority=174
step 139: visit (24, 12)   cost=132  estimate=44  priority=176
step 140: visit (24, 11)   cost=133  estimate=45  priority=178
step 141: visit (24, 10)   cost=134  estimate=46  priority=180
step 142: visit (23, 10)   cost=135  estimate=47  priority=182
step 143: visit (22, 10)   cost=136  estimate=48  priority=184
step 144: visit (22, 9)   cost=137  estimate=49  priority=186
step 145: visit (22, 8)   cost=138  estimate=50  priority=188
step 146: visit (22, 7)   cost=139  estimate=51  priority=190
step 147: visit (22, 6)   cost=140  estimate=52  priority=192
step 148: visit (23, 6)   cost=141  estimate=51  priority=192
step 149: visit (24, 6)   cost=142  estimate=50  priority=192
step 150: visit (24, 7)   cost=143  estimate=49  priority=192
step 151: visit (24, 8)   cost=144  estimate=48  priority=192
step 152: visit (25, 8)   cost=145  estimate=47  priority=192
s

step 269: visit (34, 18)   cost=224  estimate=28  priority=252
step 270: visit (28, 2)   cost=202  estimate=50  priority=252
step 271: visit (29, 2)   cost=203  estimate=49  priority=252
step 272: visit (30, 2)   cost=204  estimate=48  priority=252
step 273: visit (30, 3)   cost=205  estimate=47  priority=252
step 274: visit (30, 4)   cost=206  estimate=46  priority=252
step 275: visit (20, 4)   cost=196  estimate=56  priority=252
step 276: visit (34, 17)   cost=225  estimate=29  priority=254
step 277: visit (37, 8)   cost=219  estimate=35  priority=254
step 278: visit (31, 10)   cost=215  estimate=39  priority=254
step 279: visit (27, 2)   cost=203  estimate=51  priority=254
step 280: visit (20, 3)   cost=197  estimate=57  priority=254
step 281: visit (34, 16)   cost=226  estimate=30  priority=256
step 282: visit (35, 16)   cost=227  estimate=29  priority=256
step 283: visit (36, 16)   cost=228  estimate=28  priority=256
step 284: visit (36, 17)   cost=229  estimate=27  priority=256
s

step 401: visit (17, 4)   cost=241  estimate=59  priority=300
step 402: visit (18, 4)   cost=242  estimate=58  priority=300
step 403: visit (18, 5)   cost=243  estimate=57  priority=300
step 404: visit (18, 6)   cost=244  estimate=56  priority=300
step 405: visit (19, 6)   cost=245  estimate=55  priority=300
step 406: visit (20, 6)   cost=246  estimate=54  priority=300
step 407: visit (20, 7)   cost=247  estimate=53  priority=300
step 408: visit (20, 8)   cost=248  estimate=52  priority=300
step 409: visit (4, 4)   cost=228  estimate=72  priority=300
step 410: visit (26, 29)   cost=277  estimate=25  priority=302
step 411: visit (19, 8)   cost=249  estimate=53  priority=302
step 412: visit (15, 4)   cost=241  estimate=61  priority=302
step 413: visit (26, 28)   cost=278  estimate=26  priority=304
step 414: visit (27, 28)   cost=279  estimate=25  priority=304
step 415: visit (28, 28)   cost=280  estimate=24  priority=304
step 416: visit (18, 8)   cost=250  estimate=54  priority=304
step 

step 533: visit (16, 28)   cost=324  estimate=36  priority=360
step 534: visit (15, 28)   cost=325  estimate=37  priority=362
step 535: visit (14, 28)   cost=326  estimate=38  priority=364
step 536: visit (14, 29)   cost=327  estimate=37  priority=364
step 537: visit (14, 30)   cost=328  estimate=36  priority=364
step 538: visit (13, 30)   cost=329  estimate=37  priority=366
step 539: visit (12, 30)   cost=330  estimate=38  priority=368
step 540: visit (11, 30)   cost=331  estimate=39  priority=370
step 541: visit (10, 30)   cost=332  estimate=40  priority=372
step 542: visit (10, 31)   cost=333  estimate=39  priority=372
step 543: visit (10, 32)   cost=334  estimate=38  priority=372
step 544: visit (11, 32)   cost=335  estimate=37  priority=372
step 545: visit (12, 32)   cost=336  estimate=36  priority=372
step 546: visit (12, 33)   cost=337  estimate=35  priority=372
step 547: visit (12, 34)   cost=338  estimate=34  priority=372
step 548: visit (12, 35)   cost=339  estimate=33  prior

step 665: visit (0, 36)   cost=368  estimate=44  priority=412
step 666: visit (38, 31)   cost=403  estimate=11  priority=414
step 667: visit (29, 34)   cost=397  estimate=17  priority=414
step 668: visit (0, 35)   cost=369  estimate=45  priority=414
step 669: visit (38, 30)   cost=404  estimate=12  priority=416
step 670: visit (28, 34)   cost=398  estimate=18  priority=416
step 671: visit (28, 35)   cost=399  estimate=17  priority=416
step 672: visit (28, 36)   cost=400  estimate=16  priority=416
step 673: visit (0, 34)   cost=370  estimate=46  priority=416
step 674: visit (1, 34)   cost=371  estimate=45  priority=416
step 675: visit (2, 34)   cost=372  estimate=44  priority=416
step 676: visit (38, 29)   cost=405  estimate=13  priority=418
step 677: visit (27, 34)   cost=399  estimate=19  priority=418
step 678: visit (2, 33)   cost=373  estimate=45  priority=418
step 679: visit (38, 28)   cost=406  estimate=14  priority=420
step 680: visit (39, 28)   cost=407  estimate=13  priority=42

In [10]:
# ---- Given: does the estimate (heuristic) really matter? (no window opens) ----
# We run YOUR search three times with three different estimates and compare them.

# Two other estimates to compare against your Manhattan heuristic:
def no_estimate(a, b):
    return 0                         # always guesses 0 -> A* turns into a blind search

def triple_estimate(a, b):
    # 3x the Manhattan distance, written out here so it does not depend on `heuristic`
    return 3 * (abs(a[0] - b[0]) + abs(a[1] - b[1]))

def run_search_with(estimate, maze, start, goal):
    """Temporarily use `estimate` as the heuristic, run the search, then restore it.
    Returns (cells_visited, path_length)."""
    global heuristic
    original = heuristic
    heuristic = estimate             # astar/pick_lowest_priority will now use this
    steps = astar(maze, start, goal)
    heuristic = original             # put the real heuristic back
    explored = steps[-1][0]
    path = steps[-1][-1]
    return len(explored), (len(path) - 1 if path else None)

# A small maze chosen so the differences are easy to see.
demo_maze = [
    [0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0],
    [1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0],
    [0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0],
    [0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0],
    [0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1],
    [0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0],
    [0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0],
    [1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0],
    [0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
]
demo_start, demo_goal = (0, 0), (12, 18)

print(f"{'estimate used':<24}{'cells visited':>14}{'path length':>14}")
print("-" * 52)
for name, estimate in [("no estimate (blind)", no_estimate),
                       ("Manhattan (yours)",   heuristic),
                       ("3x Manhattan",        triple_estimate)]:
    visited, length = run_search_with(estimate, demo_maze, demo_start, demo_goal)
    print(f"{name:<24}{visited:>14}{str(length):>14}")

print("\nWhat to notice:")
print(" - No estimate visits the MOST cells (it has no idea where the goal is),")
print("   but the path it finds is the shortest possible.")
print(" - Manhattan visits far fewer cells and finds the SAME shortest path:")
print("   a good estimate makes the search faster without hurting the result.")
print(" - 3x Manhattan visits few cells but its path is LONGER: an estimate that")
print("   is too big makes A* rush to the goal and miss the shortest route.")


estimate used            cells visited   path length
----------------------------------------------------
no estimate (blind)                140            32
Manhattan (yours)                   53            32
3x Manhattan                        40            36

What to notice:
 - No estimate visits the MOST cells (it has no idea where the goal is),
   but the path it finds is the shortest possible.
 - Manhattan visits far fewer cells and finds the SAME shortest path:
   a good estimate makes the search faster without hurting the result.
 - 3x Manhattan visits few cells but its path is LONGER: an estimate that
   is too big makes A* rush to the goal and miss the shortest route.


## Think about it (answer briefly in your report)

1. **The core rule.** In the animation, cells turn blue in a certain order. In your own
   words, how does A\* decide *which* cell to visit next? Which given function does that job?
   
A* decides which cell to explore next by always selecting the known cell with the lowest overall priority value. In the provided code, the function responsible for extracting this cell is pick_lowest_priority()   
       
2. **cost vs. estimate.** During the search you saw `cost`, `estimate` and `priority`
   printed for each cell. What is the difference between `cost` and `estimate`, and how do
   they combine into `priority`?
   
  The cost represents the exact number of steps taken from the starting point to reach the current cell. The estimate is the heuristic guess (using Manhattan distance) of how many steps remain from the current cell to the goal, ignoring any walls. They are combined via simple addition (priority = cost + estimate) to represent the total expected length of a path that passes through that specific cell.

3. **A blind search.** In the comparison, "no estimate" visited many more cells than
   Manhattan, yet found a path of the same length. Why does having *no* estimate make the
   search visit so many extra cells?
   
 Setting the estimate to zero removes the algorithm's sense of direction. Without a heuristic pulling it toward the goal, it expands equally in all possible directions based solely on the cost incurred. This exhaustive checking forces it to visit many extra cells, but it still finds the shortest path because it guarantees that no shorter route was left unexplored.

4. **Too big an estimate.** "3x Manhattan" visited very few cells but returned a *longer*
   path. Explain in your own words why over-estimating the distance can make A\* miss the
   shortest route.
   
 When the estimate is too big, the algorithm just rushes to the goal and ignores the real cost of the steps. Because of this rush, it easily gets stuck behind walls and misses the shortest route, ending up with a longer path.

5. **Try it yourself.** Change `start` and `goal` in a maze cell (pick any white cells),
   or edit a maze to add/remove walls, and rerun. What happens to the number of cells
   visited and the path length?
   
Modifying the walls or changing the start and goal positions completely alters the search space. Removing walls or moving the points closer together simplifies the maze, resulting in fewer visited cells and a shorter path. Adding walls creates dead ends, forcing the algorithm to evaluate more alternative routes, which drastically increases the number of visited cells and often extends the path length.
